# ESZA019 — Visão Computacional
## Laboratório 8 — Rastreamento de Objetos (*Object Tracking*)

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Ctrl+C, Ctrl+V e Fé

### Integrantes
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** _(preencher)_
**Data de publicação do relatório:** _(preencher)_

---
> **Declaração de uso de IA Generativa (Portaria CNPq nº 2664/2026, item c).** Utilizou-se a ferramenta
> **Claude (Anthropic)** para **apoio à redação da fundamentação teórica, comentários do código e
> organização do texto**. O conteúdo foi revisado, executado e validado pelos autores, responsáveis
> integrais pelo material (itens d e f).

## Sumário
1. Introdução
2. Fundamentação Teórica
3. Procedimentos Experimentais
4. Análise e Discussão
5. Conclusões
6. Referências

## 1. Introdução

Este relatório trata do **rastreamento de objetos** (*object tracking*): dado um objeto marcado em um
quadro inicial, o algoritmo estima a posição desse **mesmo** objeto nos quadros seguintes, mantendo sua
identidade ao longo do tempo. Foram desenvolvidos dois programas em Python/OpenCV: **(1)** rastreamento em
**vídeo de arquivo** (usando vídeos da equipe, inclusive os do trabalho de vídeo) com **seleção manual de
ROI**, exibição em tela e **gravação** do resultado; e **(2)** a versão para **webcam ao vivo**, que mostra
a imagem e o rastreamento em tempo real e também **salva** o vídeo. Comparam-se algoritmos clássicos do
OpenCV (CSRT, KCF, MOSSE) e o método baseado em *deep learning* **GOTURN**.

## 2. Fundamentação Teórica

### 2.1 Detecção vs. rastreamento
**Detecção** localiza objetos **quadro a quadro**, sem memória. **Rastreamento** recebe a posição inicial
(ROI) e **prediz** a posição nos quadros seguintes usando o histórico, o que é **mais rápido** (não
re-varre a imagem inteira) e **mantém a identidade** do objeto mesmo sob oclusões parciais. Na prática,
detecção e rastreamento são complementares (detecta-se de tempos em tempos e rastreia-se no intervalo).

### 2.2 Algoritmos de rastreamento do OpenCV
- **BOOSTING / MIL** — clássicos baseados em aprendizado *online* de um classificador; hoje superados.
- **KCF** (*Kernelized Correlation Filters*) — usa filtros de correlação no domínio da frequência;
  **rápido** e razoavelmente preciso, mas não recupera bem após oclusão total.
- **MOSSE** (*Minimum Output Sum of Squared Error*) — filtro de correlação **muito rápido** (centenas de
  FPS), robusto a variação de iluminação; menos preciso que o CSRT.
- **CSRT** (*Channel and Spatial Reliability Tracking*) — filtro de correlação com mapa de confiabilidade
  espacial; **mais preciso** (lida com objetos não retangulares e mudança de escala), porém mais lento.
- **MedianFlow** — bom quando o movimento é suave e previsível; detecta falhas de rastreamento.
- **GOTURN** (*Generic Object Tracking Using Regression Networks*) — **rede neural convolucional** treinada
  *offline* que **regride** a nova caixa a partir de dois recortes (quadro anterior e atual). É o único
  baseado em *deep learning* do módulo e exige os arquivos do modelo (`goturn.prototxt` e
  `goturn.caffemodel`).

### 2.3 Compromissos (velocidade × robustez)
Há um compromisso claro: **MOSSE/KCF** priorizam **velocidade**; **CSRT** prioriza **precisão**; **GOTURN**
generaliza para objetos não vistos, mas depende de GPU/modelo e pode falhar sob domínios muito diferentes
do treino. A escolha depende da aplicação (tempo real embarcado vs. precisão).

### 2.4 Métricas e falhas típicas
Avaliam-se **acurácia** (sobreposição/IoU com o objeto real), **robustez** (número de perdas/re-inits) e
**velocidade** (FPS). Falhas comuns: **oclusão total**, **mudança brusca de escala/iluminação**,
**objetos semelhantes** ao redor e **saída do campo de visão**.

## 3. Procedimentos Experimentais

### 3.1 Ambiente
```bash
cd "Laboratório 8"
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt   # opencv-contrib-python, numpy
```

### 3.2 Programas desenvolvidos
- `rastreamento_video.py` — Experimento (1): vídeo de arquivo, seleção manual de ROI, exibe e **grava** o
  resultado.
- `rastreamento_webcam.py` — Experimento (2): webcam ao vivo, janela em tempo real e **gravação**.

A função-chave em ambos, comum a todos os métodos:

```python
tracker = cv2.TrackerCSRT_create()   # ou KCF/MOSSE/GOTURN (via --metodo)
tracker.init(frame, bbox)            # bbox = ROI selecionada com cv2.selectROI
ok, bbox = tracker.update(frame)     # a cada quadro seguinte
```

### 3.3 Execução — Experimento 1 (vídeo)
```bash
python3 rastreamento_video.py --video <seu_video>.mp4 --metodo CSRT --saida saida_rastreada.mp4
```
**_(inserir: 2–3 prints do rastreamento e o link/arquivo `saida_rastreada.mp4`)_**

### 3.4 Execução — Experimento 2 (webcam)
```bash
python3 rastreamento_webcam.py --camera 0 --metodo CSRT --saida webcam_rastreada.mp4
```
**_(inserir: print da janela ao vivo e o arquivo `webcam_rastreada.mp4`)_**

### 3.5 (Opcional) GOTURN — *deep learning*
Baixe `goturn.caffemodel` (link do roteiro) e o `goturn.prototxt` para a pasta do Lab 8 e rode
`--metodo GOTURN`. **_(inserir resultado, se testado)_**

### 3.6 Célula auxiliar — exibir um quadro do resultado dentro do relatório
Rode após gerar os vídeos, para incorporar uma imagem ao notebook.

In [ ]:
import cv2, matplotlib.pyplot as plt

# Ajuste o caminho para um dos videos gerados por voces:
video = "saida_rastreada.mp4"     # ou "webcam_rastreada.mp4"
cap = cv2.VideoCapture(video)
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)   # pega ~o 30o quadro
ok, frame = cap.read(); cap.release()

if ok:
    plt.figure(figsize=(8,5))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f"Quadro do rastreamento — {video}"); plt.show()
else:
    print("Gere primeiro o video de saida rodando rastreamento_video.py / rastreamento_webcam.py")

## 4. Análise e Discussão

Comparação qualitativa dos métodos testados (preencha com o que observaram):

| Método | Velocidade (FPS) | Precisão | Comportamento sob oclusão | Observações da equipe |
|--------|------------------|----------|---------------------------|-----------------------|
| CSRT   | _(medido)_ | alta | _(preencher)_ | _(preencher)_ |
| KCF    | _(medido)_ | média | _(preencher)_ | _(preencher)_ |
| MOSSE  | _(medido)_ | menor | _(preencher)_ | _(preencher)_ |
| GOTURN | _(medido)_ | — | _(preencher)_ | _(se testado)_ |

Pontos a discutir: (i) o **compromisso velocidade × precisão** observado (MOSSE rápido porém menos preciso;
CSRT preciso porém mais lento); (ii) **quando o rastreador perdeu o objeto** (oclusão, saída de quadro,
mudança de escala/iluminação) e como cada método reagiu; (iii) diferença entre rodar em **vídeo gravado**
(taxa estável) e **ao vivo** (variação de FPS e iluminação); (iv) impacto da **qualidade da ROI inicial**.
**_(inserir a análise com base nos vídeos de vocês.)_**

## 5. Conclusões

Foram implementados e validados dois sistemas de rastreamento (vídeo e webcam) com seleção manual de ROI e
gravação dos resultados. Os experimentos evidenciaram o **compromisso entre velocidade e precisão** entre
os algoritmos do OpenCV: _(resumir o método que melhor atendeu ao caso de vocês e por quê)_. O rastreamento
mostrou-se **mais eficiente que a redetecção quadro a quadro** e adequado a aplicações em tempo real,
respeitadas as limitações sob oclusão e mudanças bruscas de aparência. _(Comente resultados reais.)_

## 6. Referências
1. Introduction to OpenCV Tracker — https://docs.opencv.org/4.x/d2/d0a/tutorial_introduction_to_tracker.html
2. Object Tracking using OpenCV (C++/Python) — https://learnopencv.com/object-tracking-using-opencv-cpp-python/
3. GOTURN: Deep Learning based Object Tracking — https://learnopencv.com/goturn-deep-learning-based-object-tracking/
4. HELD, D.; THRUN, S.; SAVARESE, S. *Learning to Track at 100 FPS with Deep Regression Networks (GOTURN)*, ECCV 2016.
5. Roteiro do Laboratório 8 — ESZA019 (2026.2), Prof. Celso S. Kurashima.
6. CNPq. Portaria 2664/2026 (integridade e uso de IAG).